In [2]:
###############################
# Goal of this code is to conduct moisture budget analysis and calculate Q2 in 4D
# Output: Q2, dq/dt, udq/dx, vdq/dy, wdq/dp 
# Input: q, u, v, w
# use Q2 = -[ dq/dt + udq/dx + vdq/dy + wdq/dp ]
# 2024.4.21
# Mu-Ting Chien
################################################

In [3]:
import os
import sys
sys.path.append('/glade/work/muting/function/')
import KW_diagnostics as KW
import mjo_mean_state_diagnostics as MJO
from netCDF4 import Dataset
import numpy as np
import matplotlib.pyplot as plt

/glade/u/ssg/ch/usr/jupyterhub/envs/cmip6-201910a/lib/python3.7/site-packages/statsmodels/tools/_testing.py:19: FutureWarning: pandas.util.testing is deprecated. Use the functions in the public API at pandas.testing instead.
  import pandas.util.testing as tm


In [61]:
dir_out            = '/glade/work/muting/KW/'
CASENAME_LIST2     = list(['SST_AQP3_Qobs_27_-4K',\
                          'SST_AQP3_Qobs_27',\
                          'SST_AQP3_Qobs_27_4K'])
CASENAME_SHORT_LIST = list(['-4K','CTL','4K'])

icase  = 0 # 0-2 (-4K, CTL, or +4K)
nudge  = 3 # 0-3
inudge = 4 # 0-4 change this!!! from 4 to 1!

if nudge == 0:
    
    if icase != 1:
        CASENAME       = CASENAME_LIST2[icase]+'_3h_20y_CLUBB_new'
    else:
        CASENAME       = CASENAME_LIST2[icase]+'_3h_20y_CLUBB_new_uv_tendency'
    figdir         = dir_out+'figure/Post_general/Test_CLUBB/'+CASENAME_SHORT_LIST[icase]+'/'
    CASENAME_SHORT = CASENAME_SHORT_LIST[icase]+'_CLUBB_new'
elif nudge == 1:
    NUDGE_LIST = list(['3h_20y_CLUBB_new','nudge_qlf_30m','nudge_qbl_30m','nudge_uvbl_30m','nudge_Tlf_30m',]) #'nudge_qlf_30m',#'nudge_uvbl'
    title = list(['(a) No nudging','(b) q 850-700 hPa','(c) q 1000-850 hPa','(d) uv 1000-850 hPa','(e) T 850-700 hPa'])
    # note thaat the above title are all 30 minute nudging timescale, so I do not include that
elif nudge == 2:
    NUDGE_LIST = list(['3h_20y_CLUBB_new','nudge_Tlf_4d','nudge_Tlf_2d','nudge_Tlf_1d','nudge_Tlf_12h']) #'nudge_qlf_30m',#'nudge_uvbl'
    title = list(['(a) No nudging','(b) Tlf 4d','(c) Tlf 2d','(d) Tlf 1d','(e) Tlf 12h'])    
elif nudge == 3:
    NUDGE_LIST = list(['3h_20y_CLUBB_new','nudge_qlf_4d','nudge_qlf_2d','nudge_qlf_1d','nudge_qlf_12h']) #'nudge_qlf_30m',#'nudge_uvbl'
    title = list(['(a) No nudging','(b) qlf 4d','(c) qlf 2d','(d) qlf 1d','(e) qlf 12h'])

if nudge != 0:
    n_nudge = np.size(NUDGE_LIST)
    CASENAME       = CASENAME_LIST2[icase]+'_'+NUDGE_LIST[inudge]
    print(CASENAME)
        
    if inudge == 0:
        CASENAME_SHORT = CASENAME_SHORT_LIST[icase]+'_CLUBB_new'
    else:
        CASENAME_SHORT = CASENAME_SHORT_LIST[icase]+'_'+NUDGE_LIST[inudge]
    figdir         = dir_out+'figure/Post_general/Nudging/'+CASENAME_SHORT+'/'
    print(CASENAME_SHORT)

SST_AQP3_Qobs_27_-4K_nudge_qlf_12h
-4K_nudge_qlf_12h


In [62]:
os.makedirs(figdir, exist_ok=True)
output_dir = dir_out+'output_data/'+CASENAME+'/'
os.makedirs(output_dir, exist_ok=True)
s2d = 86400
Cp = 1004.64

iyr_min = 2
iyr_max = 2 #2-4
nyr = iyr_max-iyr_min+1
yr = np.arange(0,nyr)+1
latmax = 12.5
test_val = 1

Fs = 8 # how many data per day (Fs=8 is equivalent to 3-hourly data)
dt = s2d/Fs

In [63]:
# Load q
nfile_skip = KW.find_nfile_skip(CASENAME, CASENAME_SHORT, iyr_min, iyr_max, from_cheyenne=0)

vname = list(['Q'])
q, q_m, time, plev, lon, lat = KW.load_3D_data_as_1variable(CASENAME, CASENAME_SHORT, vname, iyr_min, iyr_max, latmax, nfile_skip, from_cheyenne=0, kw_proj=0)

# discard the 1000 hPa because it may contain np.nan
q = q[:,:-1,:,:]
plev = plev[:-1]

# Change unit of plev into (Pa)
if np.max(plev)<1200:
    plev = plev*100 # plev(Pa)
    print('already changed pressure unit') 

if np.sum(np.isnan(q))!=0:
    print('Caution: There is still nan 1 layer above 1000 hPa!')

nt = np.size(time)
nlev = np.size(plev)
nlon = np.size(lon)
nlat = np.size(lat)

already changed pressure unit


In [64]:
#**********************************************
#           Calculate Q2  
# Q2 = -( d(q)/d(t) + udq/dx  + vdq/dy + wdq/dp  )
#           [0]           [1]       [2]   [3]  
# d means partial, w means omega
#*****************TERM [0]********************
# partial(q)/partial(t)
dqdt = np.ones([nt-2,nlev,nlat,nlon]) #(time-2,lev-1,lat,lon)

for ii in range(0,nlev): #lev
    q_temp = q[:,ii,:,:]
    for tt in range(1,nt-1):
        dq_time = q_temp[tt+1,:,:] - q_temp[tt-1,:,:]
        dqdt[tt-1,ii,:,:] = dq_time/(2*dt)
    del q_temp,dq_time
dqdt = (dqdt[:,:-1,1:-1,:]+dqdt[:,1:,1:-1,:])/2
if test_val == 1:
    print('dqdt=')
    print( np.mean(np.mean(np.mean(dqdt,3),2),0) )
Q = 0
Q = Q+dqdt
print('finish term 0: dq/dt')
np.savez(output_dir+'Q2.npz',dqdt=dqdt)

dqdt=
[-3.06550678e-15  4.35602381e-16  4.02758178e-15  4.46932868e-15
  4.72797337e-15  8.75456095e-15  1.78965606e-14  5.24272769e-14
  1.39725097e-13  2.69358591e-13  3.91129516e-13  4.93667133e-13
  8.11072003e-13  1.32474318e-12  1.82094159e-12  2.30583830e-12
  2.76400694e-12  2.86262664e-12  2.61729230e-12  2.37904427e-12
  2.58042328e-12  3.30829463e-12  4.11152298e-12  4.53827356e-12
  3.95469854e-12  2.74311416e-12  1.58840992e-12  6.98927408e-13
  3.46237165e-14 -6.06624773e-13 -1.22934925e-12 -1.86132448e-12
 -1.98596402e-12 -1.38460799e-12 -8.63789807e-13 -1.02180663e-12
 -1.11121332e-12 -8.80146636e-13]
finish term 0: dq/dt


In [65]:
#*****************TERM [1]********************
# <u*dq/dx>
vname = list(['U'])
U, U_m, time, plev_org, lon, lat = KW.load_3D_data_as_1variable(CASENAME, CASENAME_SHORT, vname, iyr_min, iyr_max, latmax, nfile_skip, from_cheyenne=0, kw_proj=0)

# discard the 1000 hPa because it may contain np.nan
U = U[:,:-1,:,:]

dx = (lon[3]-lon[1])*np.cos(lat*2*np.pi/360)*111000 # convert into meters
dqdx = np.ones([nlev,nlat,nlon])
udqdx = np.ones([nt,nlev,nlat,nlon])
for tt in range(0,nt): #try: faster to have this loop or not, this can be removed too
    q_temp = q[tt,:,:,:]
    for ii in range(0,nlon):
        if ii == 0:
             dqdx[:,:,ii] = q_temp[:,:,1] - q_temp[:,:,143]
        elif ii == 143:
             dqdx[:,:,ii] = q_temp[:,:,0] - q_temp[:,:,142]
        else:
             dqdx[:,:,ii] = q_temp[:,:,ii+1] - q_temp[:,:,ii-1]

    for ilat in range(0,nlat):
        dqdx[:,ilat,:] = dqdx[:,ilat,:]/dx[ilat]
    udqdx[tt,:,:,:] = dqdx*U[tt,:,:,:]
    
del dqdx, U
udqdx = ( udqdx[1:-1,:-1,1:-1,:]+udqdx[1:-1,1:,1:-1,:] )/2
Q = Q+udqdx
if test_val == 1:
    print('udqdx=')
    print( np.mean(np.mean(np.mean(udqdx,3),2),0) )
print('finish term 1: u*dq/dx')   
np.savez(output_dir+'Q2.npz',dqdt=dqdt, udqdx=udqdx)

udqdx=
[-6.32531467e-15 -9.88427031e-15 -2.01235275e-14 -6.10415339e-14
 -4.81797543e-13 -3.69097371e-12 -1.39384970e-11 -2.94782003e-11
 -4.39893825e-11 -5.15018009e-11 -6.02956064e-11 -6.14749197e-11
 -3.78367192e-11  2.95257755e-11  1.40881589e-10  2.81056941e-10
  4.59848229e-10  6.06115133e-10  7.07264061e-10  8.24491295e-10
  8.39007753e-10  7.24934545e-10  6.07041838e-10  4.94171894e-10
  3.45723331e-10  1.96475520e-10  9.77986131e-11  9.82251968e-11
  1.77455261e-10  2.61040494e-10  2.84906937e-10  2.70765102e-10
  2.58682706e-10  2.37661140e-10  2.24833470e-10  1.96627657e-10
  1.69348516e-10  2.21385283e-10]
finish term 1: u*dq/dx


In [66]:
#*****************TERM [2]********************
# <v*dq/dy>
vname = list(['V'])
V, V_m, time, plev_org, lon, lat = KW.load_3D_data_as_1variable(CASENAME, CASENAME_SHORT, vname, iyr_min, iyr_max, latmax, nfile_skip, from_cheyenne=0, kw_proj=0)

# discard the 1000 hPa because it may contain np.nan
V = V[:,:-1,:,:]
    
dy = (lat[2]-lat[0])*111000 # convert into meters
dqdy = np.ones([nlev,nlat-2,nlon])
vdqdy = np.ones([nt,nlev,nlat-2,nlon])
for tt in range(0,nt): #try: faster to have this loop or not, this can be removed too
    q_temp = q[tt,:,:,:]
    for ii in range(1,nlat-1):
         dqdy[:,ii-1,:] = q_temp[:,ii+1,:] - q_temp[:,ii-1,:]
    vdqdy[tt,:,:,:] = dqdy*V[tt,:,1:-1,:]/dy
    del q_temp

del dqdy, V
vdqdy = ( vdqdy[1:-1,:-1,:,:]+vdqdy[1:-1,1:,:,:] )/2
if test_val == 1:
    print('vdqdy=')
    print( np.mean(np.mean(np.mean(vdqdy,3),2),0) )
Q = Q+vdqdy
print('finish term 2: v*dq/dy')
np.savez(output_dir+'Q2.npz',dqdt=dqdt, udqdx=udqdx, vdqdy=vdqdy)

vdqdy=
[ 3.76073313e-15  4.28072506e-15  7.68906289e-15  9.16023386e-15
 -1.50903088e-13 -1.45281718e-12 -5.92816682e-12 -2.08903174e-11
 -5.93074390e-11 -1.06658738e-10 -1.77898291e-10 -2.42493194e-10
 -3.05989956e-10 -3.10607369e-10 -2.21904077e-10 -7.83213838e-11
  1.99859024e-10  4.53179304e-10  6.01948659e-10  7.50315288e-10
  8.95305988e-10  1.06860596e-09  1.30416069e-09  1.52167743e-09
  1.52933333e-09  1.38777406e-09  1.27024570e-09  8.54458551e-10
  2.47894418e-10 -1.08768322e-10 -2.78426754e-11  2.61448731e-10
  4.84703350e-10  8.11366193e-10  1.43366827e-09  2.04130035e-09
  2.54786724e-09  3.57548954e-09]
finish term 2: v*dq/dy


In [67]:
#*****************TERM [3]********************
# Use mid-point instead of center difference
# <omega*dq/dp>
vname = list(['OMEGA'])
W, W_m, time, plev_org, lon, lat = KW.load_3D_data_as_1variable(CASENAME, CASENAME_SHORT, vname, iyr_min, iyr_max, latmax, nfile_skip, from_cheyenne=0, kw_proj=0)

# discard the 1000 hPa because it may contain np.nan
W = W[:,:-1,:,:]

dP = plev[1:]-plev[:nlev-1]
dqdp = np.ones([nt,nlev-1,nlat-2,nlon])
Wmid = ( W[:,:-1,1:-1,:] + W[:,1:,1:-1,:] )/2
del W
for ii in range(0,nlev-1): #Takes the midpoint of each pressure level
    dqdp[:,ii,:,:] = ( q[:,ii+1,1:-1,:] - q[:,ii,1:-1,:])/dP[ii] 

wdqdp = dqdp[1:-1,:,:,:]*Wmid[1:-1,:,:,:]
if test_val == 1:
    print('wdqdp=')
    print( np.mean(np.mean(np.mean(wdqdp,3),2),0) )
Q = Q+wdqdp
print('finish term 3: w*dq/dP')
np.savez(output_dir+'Q2.npz',dqdt=dqdt, udqdx=udqdx, vdqdy=vdqdy, wdqdp=wdqdp, Q2=-Q,\
         time=time[1:-1], plev=(plev[1:]+plev[:-1])/2/100, lat=lat[1:-1], lon=lon)

wdqdp=
[ 7.55495524e-15  3.86816648e-15  3.77666542e-15  4.94460712e-15
  1.42460598e-14 -3.43259196e-12 -2.65532511e-11 -1.34462506e-10
 -3.75297032e-10 -6.15582492e-10 -1.44293841e-09 -1.65481170e-09
 -3.79722203e-09 -3.86115058e-09 -5.63925872e-09 -6.25735094e-09
 -6.04128995e-09 -8.17824917e-09 -7.89838333e-09 -7.60540059e-09
 -9.10981367e-09 -9.56587935e-09 -9.53637821e-09 -8.12097961e-09
 -5.22928325e-09 -5.33644971e-09 -5.40750792e-09 -4.99498238e-09
 -4.73955213e-09 -5.20985894e-09 -1.30953067e-08 -1.27557155e-08
 -1.38204976e-08 -1.40047943e-08 -1.48377330e-08 -1.07471938e-08
 -8.62401486e-09 -5.70992136e-09]
finish term 3: w*dq/dP


In [68]:
print(output_dir)

/glade/work/muting/KW/output_data/SST_AQP3_Qobs_27_-4K_nudge_qlf_12h/


In [ ]:
# Check data dimension is correct
data = np.load(output_dir+'Q2.npz')
dqdt = data['dqdt'] #(time[1:-1], plev_mid, lat[1:-1], lon)
udqdx = data['udqdx']
vdqdy = data['vdqdy']
wdqdp = data['wdqdp']
Q2    = data['Q2']
time_new = data['time']
plev_new = data['plev']
lat_new  = data['lat']
lon      = data['lon']
if np.shape(udqdx)!=np.shape(dqdt) or np.shape(vdqdy)!=np.shape(dqdt) or np.shape(wdqdp)!=np.shape(dqdt):
    print('Dimension incorrect!')
print(np.shape(dqdt))
print(np.size(time_new), np.size(plev_new), np.size(lat_new), np.size(lon))

In [ ]:
# Calculate time mean for each term so that we can plot the vertical profile to check the above calculationis correct
dqdt_m  = np.mean(np.mean(np.mean(dqdt,3),2),0)
udqdx_m = np.mean(np.mean(np.mean(udqdx,3),2),0)
vdqdy_m = np.mean(np.mean(np.mean(vdqdy,3),2),0)
wdqdp_m = np.mean(np.mean(np.mean(wdqdp,3),2),0)
Q2_m    = np.mean(np.mean(np.mean(Q2,3),2),0)

In [ ]:
# Plot vertical profile of the climatology of each budget term 
font = 7
fig,axes = plt.subplots(1,1,figsize=(3.2, 2.4),dpi=600)
plt.subplots_adjust(left=0.2,right=0.9,top=0.9,bottom=0.15,wspace=0.1)
plt.rcParams.update({'font.size': font})
plt.plot(-Q2_m,   plev_new, 'k')
plt.plot(dqdt_m,  plev_new, 'b')
plt.plot(udqdx_m, plev_new, 'y')
plt.plot(vdqdy_m, plev_new, 'g')
plt.plot(wdqdp_m, plev_new, 'r')
plt.legend(['-Q2','dqdt','udqdx','vdqdy','\u03C9dqdp'])
plt.title('Moisture budget climatology averaged within 10S-10N')
plt.xlabel('q tendency (kg/kg/s)')
plt.ylabel('Pressure (hPa)')
plt.gca().invert_yaxis()
plt.savefig(figdir+'dqdt_climatology.png',dpi=600)
plt.show()